# 04 - Baseline Model Evaluation

This notebook documents the baseline model evaluation stage. Model `.zip`
files are generated locally by the training notebooks, so this notebook loads saved
processed evaluation CSVs by default.


## Load baseline paths


In [1]:
from pathlib import Path
import os, sys, time

def find_baseline_root():
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "baseline", base / "MuJoCo_RL_Project_Final_Submission" / "baseline"]:
            if (candidate / "baseline_summary.md").exists() and (candidate / "results").exists():
                return candidate.resolve()
    raise RuntimeError("Could not locate baseline folder")

BASE_DIR = find_baseline_root()
os.chdir(BASE_DIR)
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RAW_DIR = BASE_DIR / "results" / "raw"
PROC_DIR = BASE_DIR / "results" / "processed"
FINAL_DIR = BASE_DIR / "results" / "final"
FIG_DIR = BASE_DIR / "figures"
REPORT_FIG_DIR = FIG_DIR / "report_ready"
for folder in [RAW_DIR, PROC_DIR, FINAL_DIR, REPORT_FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

ENV_IDS = ["HalfCheetah-v5", "Hopper-v5", "Walker2d-v5"]
ALGORITHMS = ["PPO", "SAC", "TD3", "DDPG", "TQC"]
print(f"Baseline root: {BASE_DIR}")
print(f"Python: {sys.executable}")


Baseline root: D:\MuJoCo_RL_Project\MuJoCo_RL_Project_Final_Submission\baseline
Python: C:\Users\digilians01\.conda\envs\RL_PROJECT\python.exe


## Evaluation configuration


In [2]:
import gymnasium as gym
import torch
from stable_baselines3 import PPO, SAC, TD3, DDPG
from sb3_contrib import TQC

RUN_MODEL_EVALUATION = False
N_EVAL_EPISODES = 20
NOISE_SIGMAS = [0.00, 0.05, 0.10, 0.20]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODELS_DIR = BASE_DIR / "models"
ALGO_LOADERS = {"PPO": PPO, "SAC": SAC, "TD3": TD3, "DDPG": DDPG, "TQC": TQC}
print(f"RUN_MODEL_EVALUATION: {RUN_MODEL_EVALUATION}")
print(f"Device: {DEVICE}")


RUN_MODEL_EVALUATION: False
Device: cuda


## Discover local model files

This is useful if baseline models are restored locally.


In [3]:
def parse_model_name(path):
    parts = path.stem.split("__")
    if len(parts) != 4:
        return None
    env_id, algo, seed_text, step_text = parts
    return {"run_id": path.stem, "environment": env_id, "algorithm": algo,
            "seed": int(seed_text.replace("seed", "")),
            "total_timesteps": int(step_text.replace("steps", "")),
            "model_path": str(path)}

entries = []
if MODELS_DIR.exists():
    for model_path in MODELS_DIR.rglob("*.zip"):
        parsed = parse_model_name(model_path)
        if parsed:
            entries.append(parsed)
models_df = pd.DataFrame(entries)
print(f"Local model files found: {len(models_df)}")
if not models_df.empty:
    display(models_df.groupby(["environment", "algorithm"]).size().reset_index(name="models"))


Local model files found: 0


## Evaluation helpers

These functions regenerate evaluation tables when local model files exist.


In [4]:
def evaluate_model(model, env_id, n_episodes, noise_sigma=0.0, seed_base=0):
    env = gym.make(env_id)
    records = []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed_base + ep)
        total_reward, steps = 0.0, 0
        terminated, truncated = False, False
        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=True)
            if noise_sigma > 0:
                noise = np.random.normal(0, noise_sigma, size=action.shape)
                action = np.clip(action + noise, env.action_space.low, env.action_space.high)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += float(reward)
            steps += 1
        records.append({"eval_episode": ep, "eval_return": total_reward,
                        "eval_episode_length": steps, "terminated": terminated,
                        "truncated": truncated})
    env.close()
    return records

def load_model(row):
    return ALGO_LOADERS[row["algorithm"]].load(row["model_path"], device=DEVICE)


## Load saved evaluation outputs


In [5]:
final_eval = pd.read_csv(PROC_DIR / "final_eval_all.csv") if (PROC_DIR / "final_eval_all.csv").exists() else pd.DataFrame()
robust_eval = pd.read_csv(PROC_DIR / "robustness_eval_all.csv") if (PROC_DIR / "robustness_eval_all.csv").exists() else pd.DataFrame()
print(f"Final evaluation rows: {len(final_eval)}")
print(f"Robustness rows: {len(robust_eval)}")


Final evaluation rows: 85
Robustness rows: 6800


## Clean evaluation summary


In [6]:
if not final_eval.empty:
    clean_summary = (final_eval.groupby(["environment", "algorithm"])
                     .agg(runs=("run_id", "nunique"), mean_return=("mean_return", "mean"),
                          iqm_return=("iqm_return", "mean"), mean_ep_len=("mean_episode_length", "mean"),
                          termination_rate=("termination_rate", "mean"))
                     .reset_index())
    display(clean_summary.round(3))


,environment,algorithm,runs,mean_return,iqm_return,mean_ep_len,termination_rate
0,HalfCheetah-v5,DDPG,5,949.818,1041.868,1000.000,0.00
1,HalfCheetah-v5,PPO,13,795.697,817.414,1000.000,0.00
2,HalfCheetah-v5,SAC,5,7.525,13.273,1000.000,0.00
3,HalfCheetah-v5,TD3,5,580.466,608.261,1000.000,0.00
4,HalfCheetah-v5,TQC,5,467.173,485.618,1000.000,0.00
5,Hopper-v5,DDPG,5,213.213,214.312,98.830,1.00
6,Hopper-v5,PPO,6,250.489,250.611,107.575,1.00
7,Hopper-v5,SAC,5,393.460,388.020,147.040,1.00
8,Hopper-v5,TD3,5,136.071,139.388,66.940,1.00
9,Hopper-v5,TQC,5,322.181,321.332,135.690,1.00


## Robustness summary


In [7]:
if not robust_eval.empty:
    robust_summary = (robust_eval.groupby(["environment", "algorithm", "noise_sigma"])["eval_return"]
                      .agg(["mean", "std", "count"]).reset_index())
    display(robust_summary.round(3).head(30))


,environment,algorithm,noise_sigma,mean,std,count
0,HalfCheetah-v5,DDPG,0.00,853.210,659.183,100
1,HalfCheetah-v5,DDPG,0.05,894.510,605.589,100
2,HalfCheetah-v5,DDPG,0.10,973.413,615.047,100
3,HalfCheetah-v5,DDPG,0.20,886.644,555.854,100
4,HalfCheetah-v5,PPO,0.00,765.606,639.590,260
5,HalfCheetah-v5,PPO,0.05,757.150,613.569,260
6,HalfCheetah-v5,PPO,0.10,724.809,584.025,260
7,HalfCheetah-v5,PPO,0.20,644.915,475.513,260
8,HalfCheetah-v5,SAC,0.00,15.739,114.732,100
9,HalfCheetah-v5,SAC,0.05,7.195,120.221,100
